# 🧠 EEG Motor Action Classification
## Notebook 2 — Model Training: EEGNet · ResNet1D · Bi-LSTM

---

This notebook trains three neural network architectures for 8-class motor EEG classification:

| Model | Architecture | Input shape | Params (approx.) |
|-------|-------------|------------|------------------|
| **EEGNet** | Temporal + Depthwise Conv | (1, 16, 500) | ~4 K |
| **ResNet1D** | Bottleneck residual blocks | (500, 16) | ~11 M |
| **Bi-LSTM** | Bidirectional LSTM + Attention | (500, 16) | ~370 K |

**Strategy**: Cross-subject split — train on S1–S48, validate on S49–S54, test on S55–S60.

### Contents
1. Data loading & preprocessing  
2. EEGNet — architecture overview + training  
3. ResNet1D — architecture overview + training  
4. Bi-LSTM — architecture overview + training  
5. Quick comparison (→ full analysis in Notebook 03)

---
## 0. Setup

In [ ]:
import sys, os, json, time
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

# Reproducibility
import random
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

from config import (
    BATCH_SIZE, EPOCHS, LEARNING_RATE, PATIENCE,
    N_CLASSES, CLASS_NAMES, CKPT_DIR, RESULTS_DIR,
)
from src.data.loader      import load_cross_subject_splits, compute_class_weights
from src.data.preprocessing import (
    preprocess, to_eegnet_format, to_sequence_format, augment
)
from src.models.eegnet import build_eegnet, get_compiled_eegnet
from src.models.resnet import build_resnet1d_lite, get_compiled_resnet
from src.models.bilstm import build_bilstm, get_compiled_bilstm
from src.utils.metrics import evaluate, per_class_metrics
from src.utils.visualization import (
    plot_training_history, plot_confusion_matrix, plot_roc_curves
)

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

print(f'✅ TensorFlow {tf.__version__}')
print(f'   GPUs available: {len(tf.config.list_physical_devices("GPU"))}')
tf.config.list_physical_devices('GPU')

---
## 1. Data Loading & Preprocessing

In [ ]:
print('Loading all subjects (cross-subject split) ...')
t0 = time.time()
splits = load_cross_subject_splits(activity_type='M', normalize=False)
print(f'Loaded in {time.time()-t0:.1f}s')

for split_name, (X, y, _) in splits.items():
    print(f'  {split_name:5s} : X={X.shape}  y={y.shape}  classes={np.unique(y)}')

In [ ]:
# Preprocessing + format conversion
def prepare(splits, fmt='eegnet', do_augment=True):
    """
    Preprocess all splits and reshape for the target model format.
    fmt: 'eegnet'   → (N, 16, 500, 1)  height=C, width=T, channels=1
         'sequence' → (N, 500, 16)      time-first
    """
    data = {}
    for name, (X_raw, y, _) in splits.items():
        X = preprocess(X_raw, do_filter=True, do_car=True, do_zscore=True)
        if name == 'train' and do_augment:
            X, y = augment(X, y, noise_std=0.05, time_shift_p=0.3)
        if fmt == 'eegnet':
            X = to_eegnet_format(X)   # (N, 16, 500, 1)
        else:
            X = to_sequence_format(X) # (N, 500, 16)
        data[name] = (X, y)
        print(f'  [{name}] X={X.shape}  y={y.shape}')
    return data

# Class weights (computed from raw training labels)
y_train_raw = splits['train'][1]
class_weights = compute_class_weights(y_train_raw)
print('
Class weights:', {CLASS_NAMES[k]: f'{v:.2f}' for k, v in class_weights.items()})

In [ ]:
# Shared callback factory
def make_callbacks(model_name, monitor='val_accuracy'):
    return [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(CKPT_DIR / f'{model_name}_best.keras'),
            monitor=monitor, save_best_only=True, verbose=0,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor=monitor, patience=PATIENCE,
            restore_best_weights=True, verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=PATIENCE//2, min_lr=1e-6, verbose=1,
        ),
        tf.keras.callbacks.CSVLogger(
            str(RESULTS_DIR / f'{model_name}_history.csv'),
        ),
    ]

all_results = {}   # {model_name: {metrics, y_test, y_pred, y_prob, history}}

---
## 2. EEGNet

**Reference**: Lawhern et al. (2018). *EEGNet: a compact convolutional neural network for EEG-based brain–computer interfaces.* Journal of Neural Engineering.

### Architecture

```
Input (1, 16, 500)
    │
    ▼
Conv2D(F1=8, kernel=(1,64))          Temporal filter bank
BatchNorm
    │
    ▼
DepthwiseConv2D(kernel=(16,1), D=2)  Spatial filter (per electrode)
BatchNorm → ELU → AvgPool(1,4) → Dropout
    │
    ▼
SeparableConv2D(F2=16, kernel=(1,16)) Temporal smoothing
BatchNorm → ELU → AvgPool(1,8) → Dropout
    │
    ▼
Flatten → Dense(8, softmax)
```

**Key advantage**: Extremely compact (~4K params), designed specifically for EEG.

In [ ]:
# Prepare EEGNet data
print('Preparing data for EEGNet ...')
data_eeg = prepare(splits, fmt='eegnet', do_augment=True)
X_tr, y_tr = data_eeg['train']
X_va, y_va = data_eeg['val']
X_te, y_te = data_eeg['test']

In [ ]:
# Build & inspect model
eegnet = get_compiled_eegnet(learning_rate=LEARNING_RATE)
eegnet.summary(line_length=80)
print(f'\nTotal parameters: {eegnet.count_params():,}')

In [ ]:
# Architecture diagram
tf.keras.utils.plot_model(
    eegnet, to_file='../results/arch_eegnet.png',
    show_shapes=True, show_layer_names=True, dpi=80
)

In [ ]:
%%time
print('Training EEGNet ...')
hist_eeg = eegnet.fit(
    X_tr, y_tr,
    validation_data = (X_va, y_va),
    epochs          = EPOCHS,
    batch_size      = BATCH_SIZE,
    class_weight    = class_weights,
    callbacks       = make_callbacks('EEGNet'),
    verbose         = 1,
)

In [ ]:
# Training curves
fig = plot_training_history(hist_eeg.history, model_name='EEGNet')
plt.savefig('../results/02_eegnet_training.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Test evaluation
y_prob_eeg = eegnet.predict(X_te, verbose=0)
y_pred_eeg = np.argmax(y_prob_eeg, axis=1)

print('=== EEGNet — Test Set Results ===')
metrics_eeg = evaluate(y_te, y_pred_eeg, y_prob_eeg)
all_results['EEGNet'] = dict(
    metrics=metrics_eeg, y_test=y_te, y_pred=y_pred_eeg,
    y_prob=y_prob_eeg, history=hist_eeg.history
)

In [ ]:
fig = plot_confusion_matrix(y_te, y_pred_eeg, model_name='EEGNet')
plt.savefig('../results/02_eegnet_confusion.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
fig = plot_roc_curves(y_te, y_prob_eeg, model_name='EEGNet')
plt.savefig('../results/02_eegnet_roc.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
print('\nPer-class metrics:')
display(per_class_metrics(y_te, y_pred_eeg))

---
## 3. ResNet1D

Adapts the **ResNet-50 bottleneck architecture** to 1-D temporal EEG signals.
Residual connections allow training very deep networks without vanishing gradients.

### Architecture

```
Input (500, 16)
    │
    ▼
Stem: Conv1D(64, 7, stride=2) → BN → ELU → MaxPool(3, stride=2)
    │                                          → (~125, 64)
    ▼
Stage 1: 2 × BottleneckBlock(32→128)            [no stride]
Stage 2: 3 × BottleneckBlock(64→256, stride=2)  [×2 downsample]
Stage 3: 2 × BottleneckBlock(128→512, stride=2) [×2 downsample]
    │
    ▼
GlobalAveragePooling1D → Dropout → Dense(8, softmax)
```

**Bottleneck block** (identical to ResNet-50):
```
x → Conv1D(f, 1) → BN → ELU
  → Conv1D(f, K) → BN → ELU    (with stride for downsampling)
  → Conv1D(f_out, 1) → BN
  → Add(skip) → ELU
```

In [ ]:
# Prepare sequence data (shared by ResNet + BiLSTM)
print('Preparing data for ResNet1D ...')
data_seq = prepare(splits, fmt='sequence', do_augment=True)
X_tr_s, y_tr_s = data_seq['train']
X_va_s, y_va_s = data_seq['val']
X_te_s, y_te_s = data_seq['test']

In [ ]:
# Build ResNet1D (lite variant — practical for CPU/single GPU)
resnet = get_compiled_resnet(lite=True, learning_rate=LEARNING_RATE)
resnet.summary(line_length=80)
print(f'\nTotal parameters: {resnet.count_params():,}')

In [ ]:
tf.keras.utils.plot_model(
    resnet, to_file='../results/arch_resnet.png',
    show_shapes=True, show_layer_names=True, dpi=80
)

In [ ]:
%%time
print('Training ResNet1D ...')
hist_res = resnet.fit(
    X_tr_s, y_tr_s,
    validation_data = (X_va_s, y_va_s),
    epochs          = EPOCHS,
    batch_size      = BATCH_SIZE,
    class_weight    = class_weights,
    callbacks       = make_callbacks('ResNet1D'),
    verbose         = 1,
)

In [ ]:
fig = plot_training_history(hist_res.history, model_name='ResNet1D')
plt.savefig('../results/02_resnet_training.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
y_prob_res = resnet.predict(X_te_s, verbose=0)
y_pred_res = np.argmax(y_prob_res, axis=1)

print('=== ResNet1D — Test Set Results ===')
metrics_res = evaluate(y_te_s, y_pred_res, y_prob_res)
all_results['ResNet1D'] = dict(
    metrics=metrics_res, y_test=y_te_s, y_pred=y_pred_res,
    y_prob=y_prob_res, history=hist_res.history
)

In [ ]:
fig = plot_confusion_matrix(y_te_s, y_pred_res, model_name='ResNet1D')
plt.savefig('../results/02_resnet_confusion.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
fig = plot_roc_curves(y_te_s, y_prob_res, model_name='ResNet1D')
plt.savefig('../results/02_resnet_roc.png', bbox_inches='tight', dpi=150)
plt.show()

print('\nPer-class metrics:')
display(per_class_metrics(y_te_s, y_pred_res))

---
## 4. Bi-LSTM with Temporal Attention

**Bidirectional LSTM** captures temporal dependencies from both directions.
**Bahdanau-style soft attention** learns which time steps are most informative.

### Architecture

```
Input (500, 16)
    │
    ▼
Linear Projection → Dense(64) → LayerNorm
    │
    ▼
BiLSTM(128) → LayerNorm → Dropout(0.5)
    │
    ▼
BiLSTM(64)  → LayerNorm → Dropout(0.5)
    │
    ▼
Temporal Attention → context vector (256,)
    │
    ▼
Dense(64, elu) → Dropout → Dense(8, softmax)
```

**Attention** formula:  
score(t) = v·tanh(W·h_t)  
α(t) = softmax(score(t))  
context = Σ α(t)·h_t

In [ ]:
bilstm = get_compiled_bilstm(variant='standard', learning_rate=LEARNING_RATE)
bilstm.summary(line_length=80)
print(f'\nTotal parameters: {bilstm.count_params():,}')

In [ ]:
tf.keras.utils.plot_model(
    bilstm, to_file='../results/arch_bilstm.png',
    show_shapes=True, show_layer_names=True, dpi=80
)

In [ ]:
%%time
print('Training Bi-LSTM ...')
hist_bi = bilstm.fit(
    X_tr_s, y_tr_s,
    validation_data = (X_va_s, y_va_s),
    epochs          = EPOCHS,
    batch_size      = BATCH_SIZE,
    class_weight    = class_weights,
    callbacks       = make_callbacks('BiLSTM'),
    verbose         = 1,
)

In [ ]:
fig = plot_training_history(hist_bi.history, model_name='Bi-LSTM')
plt.savefig('../results/02_bilstm_training.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
y_prob_bi = bilstm.predict(X_te_s, verbose=0)
y_pred_bi = np.argmax(y_prob_bi, axis=1)

print('=== Bi-LSTM — Test Set Results ===')
metrics_bi = evaluate(y_te_s, y_pred_bi, y_prob_bi)
all_results['BiLSTM'] = dict(
    metrics=metrics_bi, y_test=y_te_s, y_pred=y_pred_bi,
    y_prob=y_prob_bi, history=hist_bi.history
)

In [ ]:
fig = plot_confusion_matrix(y_te_s, y_pred_bi, model_name='Bi-LSTM')
plt.savefig('../results/02_bilstm_confusion.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
fig = plot_roc_curves(y_te_s, y_prob_bi, model_name='Bi-LSTM')
plt.savefig('../results/02_bilstm_roc.png', bbox_inches='tight', dpi=150)
plt.show()

print('\nPer-class metrics:')
display(per_class_metrics(y_te_s, y_pred_bi))

---
## 5. Attention Weight Visualisation (Bi-LSTM)

In [ ]:
# Extract attention layer and build intermediate model
from src.utils.visualization import plot_attention_weights

# Build a sub-model that outputs attention weights
attn_layer = bilstm.get_layer('attention')

# Intermediate model: same input, outputs (logits, attention_weights)
intermediate = tf.keras.Model(
    inputs  = bilstm.input,
    outputs = [bilstm.output, attn_layer.call(bilstm.get_layer('drop_1').output)[1]],
)

# Run on test set CLH trials (class 1)
mask_clh = y_te_s == 1
if mask_clh.sum() > 0:
    logits, attn_weights = intermediate.predict(X_te_s[mask_clh], verbose=0)
    fig = plot_attention_weights(attn_weights, class_name='CLH')
    plt.savefig('../results/02_bilstm_attention_clh.png', bbox_inches='tight', dpi=150)
    plt.show()
else:
    print('No CLH trials in test set for this split.')

---
## 6. Quick Comparison Summary

In [ ]:
from src.utils.metrics import build_comparison_table

comparison = build_comparison_table(
    {name: r['metrics'] for name, r in all_results.items()}
)
print('\n' + '='*60)
print('  MODEL COMPARISON — TEST SET')
print('='*60)
display(comparison.style.highlight_max(axis=0, color='lightgreen'))

comparison.to_csv('../results/02_model_comparison.csv')
print('\nSaved to results/02_model_comparison.csv')

In [ ]:
# Training curve overlay
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
styles = {'EEGNet': ('steelblue', '-'), 'ResNet1D': ('coral', '--'), 'BiLSTM': ('seagreen', ':')}

for name, res in all_results.items():
    hist = res['history']
    ep   = range(1, len(hist['loss']) + 1)
    c, ls = styles[name]
    axes[0].plot(ep, hist['val_loss'],     color=c, linestyle=ls, label=name, linewidth=1.6)
    axes[1].plot(ep, hist['val_accuracy'], color=c, linestyle=ls, label=name, linewidth=1.6)

for ax, title in zip(axes, ['Val Loss', 'Val Accuracy']):
    ax.set_xlabel('Epoch'); ax.set_title(title, fontsize=12)
    ax.legend(); ax.spines[['top','right']].set_visible(False)

fig.suptitle('Validation Curves — All Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/02_all_models_curves.png', bbox_inches='tight', dpi=150)
plt.show()

---
## Summary

All three models have been trained and evaluated on the cross-subject test set.

| Model | ~Params | Strength | Weakness |
|-------|---------|----------|----------|
| EEGNet | 4 K | Ultra-compact, EEG-specific design | May underfit complex patterns |
| ResNet1D | 500 K–11 M | Deep residual learning, strong features | Slower, more data needed |
| Bi-LSTM | 370 K | Temporal context, interpretable attention | Sequential training, slow |

→ Proceed to **Notebook 03** for detailed results analysis and comparison.